# DeepSeismic2 · Volve Data Exploration

Hands-on notebook for inspecting Volve seismic data, validating the ingest pipeline,
and visualising seismic sections.

**Data required (one of):**
- Real ST10010: `data/volve/raw/ST10010ZC11_PZ_PSDM_KIRCH_FULL_T.MIG_FIN.POST_STACK.3D.JS-017536.segy`
- Synthetic sample (generated in Cell 2 below if missing)

See `docs/volve-data-acquisition.md` for download instructions.

In [ ]:
from __future__ import annotations

import subprocess
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import zarr

# Ensure project src is importable when running from notebooks/
PROJ_ROOT = Path("..").resolve()
if str(PROJ_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJ_ROOT / "src"))

from deepseismic.ingest.segy_loader import SEGYLoader, load_segy, segy_to_zarr

plt.rcParams.update({"figure.dpi": 120, "figure.figsize": (13, 5), "axes.titlesize": 11})
print("Libraries loaded OK")

In [ ]:
DATA_DIR   = PROJ_ROOT / "data" / "volve"
RAW_DIR    = DATA_DIR / "raw"
STAGED_DIR = DATA_DIR / "staged"

for d in (DATA_DIR, RAW_DIR, STAGED_DIR):
    d.mkdir(parents=True, exist_ok=True)

# Priority: real ST10010 -> synthetic sample
REAL_SEGY  = RAW_DIR / "ST10010ZC11_PZ_PSDM_KIRCH_FULL_T.MIG_FIN.POST_STACK.3D.JS-017536.segy"
SYNTH_SEGY = DATA_DIR / "synthetic_sample.segy"

if REAL_SEGY.exists():
    ACTIVE_SEGY = REAL_SEGY
    SAMPLE_MODE = True   # load first 50 inlines for interactive QC on the ~1 GB file
    print(f"Real ST10010 found: {REAL_SEGY.stat().st_size / 1e9:.2f} GB  (sample_mode=True)")
elif SYNTH_SEGY.exists():
    ACTIVE_SEGY = SYNTH_SEGY
    SAMPLE_MODE = False
    print(f"Synthetic sample found: {SYNTH_SEGY.stat().st_size / 1e6:.1f} MB")
else:
    print("No SEG-Y found. Run the next cell to generate the synthetic sample.")
    ACTIVE_SEGY = SYNTH_SEGY
    SAMPLE_MODE = False

In [ ]:
# Generate synthetic sample if no SEG-Y is present yet
if not ACTIVE_SEGY.exists():
    print("Generating synthetic sample (~45 MB)...")
    result = subprocess.run(
        [
            sys.executable,
            str(PROJ_ROOT / "scripts" / "download_volve.py"),
            "--sample",
            "--dest", str(DATA_DIR),
        ],
        capture_output=True,
        text=True,
        cwd=str(PROJ_ROOT),
    )
    print(result.stdout[-3000:] if len(result.stdout) > 3000 else result.stdout)
    if result.returncode != 0:
        print("STDERR:", result.stderr[-1000:])
    else:
        ACTIVE_SEGY = DATA_DIR / "synthetic_sample.segy"
        print(f"Created: {ACTIVE_SEGY.stat().st_size / 1e6:.1f} MB")
else:
    print(f"SEG-Y ready: {ACTIVE_SEGY.name}  ({ACTIVE_SEGY.stat().st_size / 1e6:.1f} MB)")

In [ ]:
print(f"Loading: {ACTIVE_SEGY.name}")
print(f"sample_mode={SAMPLE_MODE}  (True = first 50 inlines only)\n")

ds, geom = load_segy(str(ACTIVE_SEGY), sample_mode=SAMPLE_MODE, sample_n_inlines=50)

print("Survey Geometry")
print("=" * 44)
print(f"  Inline range :  {geom.inline_min} - {geom.inline_max}  step {geom.inline_step}")
print(f"  Crossline    :  {geom.crossline_min} - {geom.crossline_max}  step {geom.crossline_step}")
print(f"  Inlines (n)  :  {geom.n_inlines}")
print(f"  Crosslines(n):  {geom.n_crosslines}")
print(f"  Samples/trace:  {geom.n_samples}")
print(f"  Sample rate  :  {geom.sample_rate_ms} ms")
print(f"  TWT range    :  {geom.times_ms[0]:.0f} - {geom.times_ms[-1]:.0f} ms")
print(f"  Datum        :  {geom.datum_ms} ms")
print()
print("xarray Dataset:")
print(ds)

In [ ]:
amp = ds["amplitude"].values  # (n_IL, n_XL, n_T)  float32
flat = amp.ravel()
nz   = flat[flat != 0.0]

stats = {
    "Shape (IL x XL x T)": str(amp.shape),
    "dtype":               str(amp.dtype),
    "Memory (MB)":         f"{amp.nbytes / 1e6:.1f}",
    "min":                 f"{float(flat.min()):.5f}",
    "max":                 f"{float(flat.max()):.5f}",
    "mean":                f"{float(flat.mean()):.5f}",
    "std":                 f"{float(flat.std()):.5f}",
    "p01":                 f"{float(np.percentile(flat, 1)):.5f}",
    "p99":                 f"{float(np.percentile(flat, 99)):.5f}",
    "non-zero fraction":   f"{nz.size / max(flat.size, 1):.4f}",
}
df_stats = pd.DataFrame.from_dict(stats, orient="index", columns=["Value"])
display(df_stats)

In [ ]:
p01, p99 = np.percentile(flat, 1), np.percentile(flat, 99)
clipped  = np.clip(flat, p01, p99)

fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(clipped, bins=300, color="#2B6CB0", edgecolor="none", alpha=0.85)
ax.axvline(0,           color="k",   lw=0.9, ls="--", label="zero")
ax.axvline(float(flat.mean()), color="firebrick", lw=0.9, ls="--",
           label=f"mean = {float(flat.mean()):.4f}")
ax.set_xlabel("Amplitude")
ax.set_ylabel("Count")
ax.set_title(f"Amplitude histogram  |  {ACTIVE_SEGY.name}  |  clipped to [p01, p99]")
ax.legend()
plt.tight_layout()
plt.show()
print(f"Clip range: [{p01:.4f}, {p99:.4f}]")

In [ ]:
# Middle inline section (IL x T)
n_il_loaded = amp.shape[0]
inlines_loaded = geom.inlines[:n_il_loaded]
il_idx = n_il_loaded // 2
il_no  = int(inlines_loaded[il_idx])

section = amp[il_idx, :, :]   # (n_XL, n_T)
clip    = float(np.percentile(np.abs(section), 98))

fig, ax = plt.subplots(figsize=(14, 6))
im = ax.imshow(
    section.T,
    aspect="auto",
    cmap="seismic",
    vmin=-clip, vmax=clip,
    extent=[
        geom.crosslines[0], geom.crosslines[min(amp.shape[1], len(geom.crosslines)) - 1],
        geom.times_ms[-1], geom.times_ms[0],
    ],
)
ax.set_xlabel("Crossline")
ax.set_ylabel("TWT (ms)")
ax.set_title(f"Inline {il_no}  |  {ACTIVE_SEGY.name}")
plt.colorbar(im, ax=ax, label="Amplitude", fraction=0.015, pad=0.02)
plt.tight_layout()
plt.show()

In [ ]:
# Middle crossline section (IL x T)
xl_idx = amp.shape[1] // 2
xl_no  = int(geom.crosslines[xl_idx])

section_xl = amp[:, xl_idx, :]   # (n_IL_loaded, n_T)
clip_xl    = float(np.percentile(np.abs(section_xl), 98))

fig, ax = plt.subplots(figsize=(14, 6))
im = ax.imshow(
    section_xl.T,
    aspect="auto",
    cmap="seismic",
    vmin=-clip_xl, vmax=clip_xl,
    extent=[inlines_loaded[0], inlines_loaded[-1], geom.times_ms[-1], geom.times_ms[0]],
)
ax.set_xlabel("Inline")
ax.set_ylabel("TWT (ms)")
ax.set_title(f"Crossline {xl_no}  |  {ACTIVE_SEGY.name}")
plt.colorbar(im, ax=ax, label="Amplitude", fraction=0.015, pad=0.02)
plt.tight_layout()
plt.show()

In [ ]:
zarr_path = STAGED_DIR / "seismic.zarr"
print(f"Converting to Zarr: {zarr_path}")
print("(sample_mode reduces the file to first 50 inlines for a quick test)\n")

meta = segy_to_zarr(
    str(ACTIVE_SEGY),
    str(zarr_path),
    sample_mode=SAMPLE_MODE,
    sample_n_inlines=50,
    overwrite=True,
)

print("Zarr store written")
print(f"  Path    : {zarr_path}")
print(f"  Chunks  : {meta.zarr_chunks}")
n_il = meta.n_inlines_loaded
n_xl = meta.geometry['n_crosslines']
n_s  = meta.geometry['n_samples']
print(f"  Shape   : {n_il} x {n_xl} x {n_s}")
print(f"  p99 amp : {meta.amplitude_stats['p99']:.5f}")
print(f"  Sidecar : {zarr_path.with_suffix('.json')}")

In [ ]:
# Inspect the Zarr store and visualise a time slice
store = zarr.open(str(zarr_path), mode="r")
print("Zarr store tree:")
print(store.tree())

z_amp  = store["amplitude"]
z_twtt = store["twtt_ms"][:]
z_il   = store["inline"][:]
z_xl   = store["crossline"][:]

print(f"\namplitude: shape={z_amp.shape}  chunks={z_amp.chunks}  dtype={z_amp.dtype}")

# Time slice at the depth of the first synthetic reflector (~35% of TWT range)
t_idx = z_amp.shape[2] // 3
twt_ms = float(z_twtt[t_idx])
time_slice = z_amp[:, :, t_idx]

clip_ts = float(np.percentile(np.abs(time_slice), 98))
fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(time_slice.T, aspect="auto", cmap="seismic", vmin=-clip_ts, vmax=clip_ts)
ax.set_xlabel(f"Inline index (IL {z_il[0]}+)")
ax.set_ylabel(f"Crossline index (XL {z_xl[0]}+)")
ax.set_title(f"Horizontal time slice at TWT \u2248 {twt_ms:.0f} ms  (read from Zarr)")
plt.colorbar(im, ax=ax, fraction=0.025, pad=0.03)
plt.tight_layout()
plt.show()

## Next Steps

### Full ingest of the real ST10010 volume
```python
from deepseismic.ingest.segy_loader import segy_to_zarr

# ~10 min on a modern laptop; ~2 min on a cloud VM
meta = segy_to_zarr(
    "data/volve/raw/ST10010ZC11_PZ_PSDM_KIRCH_FULL_T.MIG_FIN.POST_STACK.3D.JS-017536.segy",
    "data/volve/staged/ST10010_full.zarr",
    overwrite=True,
)
print(meta.geometry, meta.amplitude_stats)
```

### Generate fault training labels
```python
from deepseismic.ingest.label_generator import (
    parse_petrel_fault_sticks, FaultMaskGenerator, SurveyTransform
)
sticks = parse_petrel_fault_sticks("data/volve/interpretations/Volve_Fault_Sticks.txt")
# ... see docs/volve-data-acquisition.md for full example
```

### Export from Databricks
```bash
export DATABRICKS_HOST=https://dbc-63d65b56-08e4.cloud.databricks.com
export DATABRICKS_TOKEN=<your-pat>
python scripts/databricks_export.py --discover
python scripts/databricks_export.py --notebook-cells  # prints notebook code
```

### Download real data
```bash
# After accepting terms at equinor.com/energy/volve-data-sharing:
python scripts/download_volve.py --seismic --base-url "https://..." --dest data/volve
```